# Nedbank Transaction Volume Forecasting - #4 Private Leaderboard Solution

**Task:** predict how many banking transactions each customer will make in the next 3 months
(November 2015 - January 2016), from ~18 million historical transactions plus financial and
demographic tables. **Metric:** RMSLE.

**Result:** #4 on the private leaderboard. Public LB 0.369841 (`panel_allcust_36seed.csv`).

This is a single, self-contained notebook: it runs top to bottom, explains every idea in plain
language, and reproduces the exact submission file. No hidden imports - the feature engine, the
metric, and the model config are all inlined below so you can read and run everything in one place.

> **TL;DR.** The one thing that makes this competition hard is that the training customers and the
> test customers are *different people*, so ordinary cross-validation misleads you. The fix is a
> **multi-anchor panel**: instead of one row per customer as of Oct 2015, we slide the "as-of" date
> backwards through history and make one row per (customer, month), each with its own realized
> next-3-month target. That turns ~8k rows into ~250k and forces the model to learn a general
> "recent behaviour -> next quarter" rule instead of memorising specific customers. On that panel we
> train one heavily-regularised **LightGBM** (L1 objective, log1p target) and average it over **36
> random seeds** to cancel noise. No GPU, no deep learning, no multi-model blend.

## 1. The problem, in plain words

Every customer swipes a card, pays a bill, or gets paid - each of those is a transaction. We are
given every transaction from Dec 2012 to Oct 2015 for a set of customers, and we must predict, for a
*different* set of customers, how many transactions each will make over the next 3 months.

We output one number per test customer. We are graded with **RMSLE** - the error is measured on a
logarithmic (squished) scale, so being off by 5 on a customer who does 200 barely matters, but being
off by 5 on a customer who does 2 matters a lot. The game is to get the quiet and medium customers
right, not the rare very-busy ones.

### The trap that defines this competition
The customers we learn from (train) and the customers we are graded on (test) are **completely
disjoint** - no overlap. So the usual trick of holding out some training customers to check
yourself measures how well you predict *people like the ones you already saw*, not *strangers* - and
strangers are what the leaderboard grades. In our experiments, for a long time **better
cross-validation produced worse leaderboard scores**. Every choice in this solution is a response to
that fact.

In [1]:
# --- Setup: imports, seeds, paths, constants (everything the notebook needs) ---
import os, gc, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings("ignore")          # silence benign empty-slice / fragmentation warnings

os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(42)                          # global seed; each model also sets its own random_state

# RUN_TRAINING = False -> fast path: load and verify the committed submission (seconds).
# RUN_TRAINING = True  -> full reproduction: rebuild the panel from raw data + train 36 seeds (hours).
RUN_TRAINING = False

DATA = Path("data")                         # place the competition files here (see data/README.md)
SUBMISSIONS = Path("submissions"); SUBMISSIONS.mkdir(exist_ok=True)
ID = "UniqueID"
TARGET = "next_3m_txn_count"
CUTOFF = pd.Timestamp("2015-10-31")         # last transaction date; nothing may post-date this
print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

lightgbm 4.6.0 | pandas 2.3.3 | numpy 2.4.3


## 2. The scoring quirk (submit log1p, not raw counts)

One non-obvious platform detail, worth stating up front because it silently ruins a first
submission: the live scorer computes plain RMSE against a reference that is stored as
`log1p(true_count)`. In effect the platform takes the logarithm of the true answers before
comparing, so we must submit `log1p(predicted_count)`, not the raw count. We proved this early: a
raw-count submission scored ~199 (nonsense), while the identical predictions submitted as `log1p`
scored ~0.38. The `write_submission` helper below applies `log1p` and validates the file format.

In [2]:
# --- Metric and submission writer (raw counts in; log1p written out) ---
def rmsle(y_true, y_pred) -> float:
    """The challenge metric. Inputs are RAW counts."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def write_submission(test_ids: pd.Series, preds: np.ndarray, name: str) -> Path:
    """Write a submission. `preds` are RAW COUNT predictions; the file stores log1p(pred), because
    the platform scores plain RMSE against a log1p(true) reference. Validates format defensively."""
    sample = pd.read_csv(DATA / "SampleSubmission.csv")
    sub = pd.DataFrame({ID: test_ids.values, TARGET: np.log1p(np.clip(preds, 0, None))})
    assert set(sub.columns) == {ID, TARGET}, "wrong columns"
    assert not sub[TARGET].isna().any(), "NaN predictions"
    assert (sub[TARGET] >= 0).all(), "negative predictions"
    assert len(sub) == len(sample), f"row count {len(sub)} != sample {len(sample)}"
    assert set(sub[ID]) == set(sample[ID]), "UniqueID set mismatch vs SampleSubmission"
    sub = sample[[ID]].merge(sub, on=ID, how="left")     # order rows as in SampleSubmission
    assert not sub[TARGET].isna().any(), "merge left NaNs - ID mismatch"
    out = SUBMISSIONS / name
    sub.to_csv(out, index=False)
    return out

## 3. Feature engineering (what we tell the model about each customer)

The function `build_asof(cutoff, ids)` below describes each customer using **only** data on or
before a given cutoff date. That strict "as-of" discipline is the key property that makes the panel
(next section) safe: the same code can be run at any historical date without leaking the future.

It produces ~530 features per customer across these families:
- **Recent-window rollups** (7/14/30/60/90/180 days): counts, sums, balances, debit/credit mix, and
  "is activity speeding up or slowing down" ratios. Recent behaviour is the strongest near-future
  signal.
- **Monthly history, momentum and slopes**: per-month counts going back years, plus rolling
  mean/std, momentum, and year-over-year ratios - trend and rhythm.
- **Seasonal / festive**: how busy this customer is specifically around Nov-Dec-Jan in prior years.
- **Recency and gaps**: days since last transaction, typical gap between transactions, quiet-month
  streaks - separates "active" from "going quiet".
- **Tenure**: true calendar months-on-book and activity density, with explicit zeros for brand-new
  customers (never the median) so the model can tell "new" from "dormant veteran".
- **Type / batch / account mix, financials, demographics**: what kinds of transactions, across how
  many accounts, plus interest-income trend, age, income, and segment.

A deliberate choice visible in the code: missing values are filled with the column median then 0.
For count/share features a missing month genuinely means "zero activity", and encoding it as 0 beat
passing NaN to the model on the leaderboard (see Section 8).

In [3]:
def _san(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

_TX_CACHE = {}
# Only the columns features_asof actually consumes (ReversalTypeDescription is unused).
# Loading a subset + downcasting shrinks the cached 18M-row table from ~7GB to ~2GB - the
# uncapped object-dtype load is what let memory balloon to ~40GB when anchors/builds overlapped.
_TX_COLS = ["UniqueID", "AccountID", "TransactionDate", "TransactionAmount",
            "TransactionTypeDescription", "TransactionBatchDescription",
            "StatementBalance", "IsDebitCredit"]


def _load_tx():
    if "tx" not in _TX_CACHE:
        T = pd.read_parquet(DATA / "transactions_features.parquet", columns=_TX_COLS)
        T["TransactionDate"] = pd.to_datetime(T["TransactionDate"])
        T["TransactionAmount"] = T["TransactionAmount"].astype("float32")
        T["StatementBalance"] = T["StatementBalance"].astype("float32")
        # low-cardinality strings -> category (13 / 8 / 2 levels): the big memory win.
        # NOT UniqueID/AccountID - they are groupby keys, and category keys trigger the
        # observed=False all-combinations explosion (esp. [ID, AccountID] in _accounts).
        for c in ("TransactionTypeDescription", "TransactionBatchDescription", "IsDebitCredit"):
            T[c] = T[c].astype("category")
        _TX_CACHE["tx"] = T
    return _TX_CACHE["tx"]


def compute_target(cutoff: pd.Timestamp) -> pd.Series:
    """Count of transactions per customer in the 3 months AFTER cutoff."""
    T = _load_tx()
    end = cutoff + pd.DateOffset(months=3)
    win = T[(T["TransactionDate"] > cutoff) & (T["TransactionDate"] <= end)]
    return win.groupby(ID).size().rename("target")


def _prep(cutoff):
    T = _load_tx()
    T = T[T["TransactionDate"] <= cutoff].copy()
    T["txn_day"] = T["TransactionDate"].dt.day
    T["txn_dow"] = T["TransactionDate"].dt.dayofweek
    T["txn_month"] = T["TransactionDate"].dt.month
    T["txn_quarter"] = T["TransactionDate"].dt.quarter
    T["is_debit"] = (T["IsDebitCredit"] == "Debit").astype("int8")
    T["is_credit"] = (T["IsDebitCredit"] == "Credit").astype("int8")
    T["is_month_end"] = (T["txn_day"] >= 25).astype("int8")
    T["is_weekend"] = (T["txn_dow"] >= 5).astype("int8")
    T["abs_amount"] = T["TransactionAmount"].abs()
    T["ym_int"] = T["TransactionDate"].dt.to_period("M").apply(lambda p: p.ordinal)
    T["days_to_cutoff"] = (cutoff - T["TransactionDate"]).dt.days
    return T


def _daily(T):
    out = None
    for days in [7, 14, 30, 60, 90, 180]:
        g = T[T["days_to_cutoff"] <= days].groupby(ID)
        w = g.agg(txn_count=("TransactionDate", "count"), txn_sum=("TransactionAmount", "sum"),
                  txn_mean=("TransactionAmount", "mean"), txn_std=("TransactionAmount", "std"),
                  txn_abs_sum=("abs_amount", "sum"), n_debit=("is_debit", "sum"),
                  n_credit=("is_credit", "sum"), n_month_end=("is_month_end", "sum"),
                  n_weekend=("is_weekend", "sum"), bal_last=("StatementBalance", "last"),
                  bal_mean=("StatementBalance", "mean"), bal_min=("StatementBalance", "min"),
                  bal_max=("StatementBalance", "max"), n_active_days=("TransactionDate", "nunique"))
        w["txn_rate"] = w["txn_count"] / days
        w["debit_ratio"] = w["n_debit"] / (w["txn_count"] + 1)
        w["active_rate"] = w["n_active_days"] / days
        w["bal_range"] = w["bal_max"] - w["bal_min"]
        w["amt_x_count"] = w["txn_abs_sum"] * w["txn_count"]
        w["net_flow"] = w["txn_sum"]
        w = w.add_prefix(f"d{days}_")
        out = w if out is None else out.join(w, how="outer")
    out = out.reset_index()
    for col in ["txn_count", "txn_abs_sum", "amt_x_count"]:
        def rate(d):
            c = out.get(f"d{d}_{col}")
            return (c.fillna(0) if c is not None else 0) / d
        out[f"{col}_accel_7_30"] = (rate(7) + 1) / (rate(30) + 1)
        out[f"{col}_accel_30_90"] = (rate(30) + 1) / (rate(90) + 1)
        out[f"{col}_accel_90_180"] = (rate(90) + 1) / (rate(180) + 1)
    return out


def _monthly(T, cutoff_int):
    m = (T.groupby([ID, "ym_int"]).agg(
        txn_count=("TransactionDate", "count"), txn_sum=("TransactionAmount", "sum"),
        txn_mean=("TransactionAmount", "mean"), txn_std=("TransactionAmount", "std"),
        txn_abs_sum=("abs_amount", "sum"), n_debit=("is_debit", "sum"),
        n_weekend=("is_weekend", "sum"), n_month_end=("is_month_end", "sum"),
        bal_mean=("StatementBalance", "mean"), bal_last=("StatementBalance", "last"),
        bal_min=("StatementBalance", "min"), bal_max=("StatementBalance", "max"),
        n_active_days=("TransactionDate", "nunique"),
        n_unique_amts=("TransactionAmount", "nunique")).reset_index())
    m["amount_cv"] = m["txn_std"].fillna(0) / (m["txn_mean"].abs() + 1e-9)
    m["debit_ratio"] = m["n_debit"] / (m["txn_count"] + 1)
    m["weekend_ratio"] = m["n_weekend"] / (m["txn_count"] + 1)
    m["month_end_ratio"] = m["n_month_end"] / (m["txn_count"] + 1)
    m["bal_range"] = m["bal_max"] - m["bal_min"]
    m["avg_daily_txns"] = m["txn_count"] / (m["n_active_days"] + 1)
    m["turnover_ratio"] = m["txn_abs_sum"] / (m["bal_mean"].abs() + 1e-9)
    m["amt_x_count"] = m["txn_abs_sum"] * m["txn_count"]
    m["net_flow"] = m["txn_sum"]
    m = m[m["ym_int"] <= cutoff_int].copy()
    m["months_ago"] = cutoff_int - m["ym_int"]
    return m


def _festive(mf):
    def wa(mas, pre):
        s = mf[mf["months_ago"].isin(mas)].groupby(ID).agg(
            count=("txn_count", "sum"), amount=("txn_abs_sum", "sum"),
            active=("n_active_days", "sum"), amt_x_count=("amt_x_count", "sum"))
        return s.add_prefix(pre)
    sw_ly = wa([9, 10, 11], "sw_ly_"); sw_2y = wa([21, 22, 23], "sw_2y_")
    q4ly = wa([10, 11, 12], "q4ly_"); q1cy = wa([7, 8, 9], "q1cy_")
    q2cy = wa([4, 5, 6], "q2cy_"); q3cy = wa([1, 2, 3], "q3cy_"); q4py = wa([22, 23, 24], "q4py_")
    qt = (sw_ly.join(sw_2y, how="outer").join(q4ly, how="outer").join(q1cy, how="outer")
          .join(q2cy, how="outer").join(q3cy, how="outer").join(q4py, how="outer")).reset_index()
    qt["ct_chng_q4_q1"] = (qt["q4ly_count"] - qt["q1cy_count"]) / (qt["q1cy_count"] + 1)
    qt["amt_chng_q4_q1"] = (qt["q4ly_amount"] - qt["q1cy_amount"]) / (qt["q1cy_amount"] + 1)
    qt["ct_chng_q3_q4"] = (qt["q3cy_count"] - qt["q4ly_count"]) / (qt["q4ly_count"] + 1)
    qt["ct_yoy_q4"] = (qt["q4ly_count"] + 1) / (qt["q4py_count"] + 1)
    qt["sw_yoy"] = (qt["sw_ly_count"] + 1) / (qt["sw_2y_count"] + 1)
    qt["festive_concentration"] = qt["q4ly_count"] / (
        qt["q4ly_count"] + qt["q1cy_count"] + qt["q2cy_count"] + qt["q3cy_count"] + 1)
    qt["predicted_festive_count"] = qt["sw_ly_count"]
    return qt


def _recency(T, mf, cutoff):
    r = T.groupby(ID).agg(last_txn=("TransactionDate", "max"), first_txn=("TransactionDate", "min"),
                          total_txns=("TransactionDate", "count")).reset_index()
    r["days_since_last"] = (cutoff - r["last_txn"]).dt.days
    r["tenure_days"] = (r["last_txn"] - r["first_txn"]).dt.days
    r["txns_per_day"] = r["total_txns"] / (r["tenure_days"] + 1)
    r["inactive_31_60"] = r["days_since_last"].between(31, 60).astype("int8")
    r["inactive_61_90"] = r["days_since_last"].between(61, 90).astype("int8")
    r["inactive_90plus"] = (r["days_since_last"] > 90).astype("int8")
    r["is_at_risk"] = r["inactive_90plus"]
    recent = mf[mf["months_ago"] <= 6]
    present = recent.groupby(ID)["months_ago"].apply(set)
    streak = {}
    for cid, s in present.items():
        k = 0
        for ma in range(0, 7):
            if ma in s:
                break
            k += 1
        streak[cid] = k
    r["consecutive_zero_months"] = r[ID].map(streak).fillna(0).astype("int8")
    return r.drop(columns=["last_txn", "first_txn"])


def _snap_mom(mf):
    MAX_LAG = 14
    cols = ["txn_count", "txn_sum", "txn_abs_sum", "bal_last", "amount_cv", "debit_ratio",
            "n_active_days", "turnover_ratio", "amt_x_count", "net_flow"]
    snap = mf[mf["months_ago"] <= MAX_LAG]
    parts = []
    for col in cols:
        piv = snap.pivot_table(index=ID, columns="months_ago", values=col, aggfunc="first")
        piv = piv.rename(columns={i: f"{col}_lag{i}" for i in range(MAX_LAG + 1)})
        parts.append(piv)
    sf = pd.concat(parts, axis=1).reset_index()

    def g(col, lag):
        c = f"{col}_lag{lag}"
        return sf[c].fillna(0) if c in sf.columns else pd.Series(0.0, index=sf.index)

    def sr(a, b):
        return (a + 1) / (b + 1)
    for col in ["txn_count", "txn_sum", "amt_x_count", "net_flow", "bal_last", "turnover_ratio"]:
        l0, l1, l2, l3, l6, l12 = (g(col, i) for i in [0, 1, 2, 3, 6, 12])
        sf[f"{col}_mom1"] = l0 - l1
        sf[f"{col}_mom3"] = l0 - l3
        sf[f"{col}_accel"] = (l0 - l1) - (l1 - l2)
        sf[f"{col}_ratio_1_3"] = sr(l0, l3)
        sf[f"{col}_ratio_3_6"] = sr(l3, l6)
        sf[f"{col}_yoy"] = sr(l0, l12)
        for w in [3, 6, 12]:
            wc = [f"{col}_lag{i}" for i in range(w) if f"{col}_lag{i}" in sf.columns]
            if wc:
                mat = sf[wc].fillna(0)
                sf[f"{col}_rmean{w}"] = mat.mean(axis=1)
                sf[f"{col}_rstd{w}"] = mat.std(axis=1).fillna(0)
                sf[f"{col}_rmax{w}"] = mat.max(axis=1)

    def slope(col, n):
        wc = [f"{col}_lag{i}" for i in range(n) if f"{col}_lag{i}" in sf.columns]
        if len(wc) < 2:
            return pd.Series(0.0, index=sf.index)
        x = np.arange(len(wc), 0, -1, dtype=float)
        x -= x.mean()
        Y = sf[wc].fillna(0).values
        return pd.Series((Y * x).sum(1) / (x ** 2).sum(), index=sf.index)
    for col in ["txn_count", "txn_abs_sum", "amt_x_count", "bal_last"]:
        for n in [3, 6, 12]:
            sf[f"{col}_slope{n}"] = slope(col, n)
    return sf


def _seasonal(T):
    q = T.groupby([ID, "txn_quarter"]).size().rename("c").reset_index()
    q["share"] = q["c"] / q.groupby(ID)["c"].transform("sum")
    qp = q.pivot_table(index=ID, columns="txn_quarter", values="share", fill_value=0)
    qp = qp.rename(columns={i: f"q{i}_share" for i in [1, 2, 3, 4]})
    mo = T.groupby([ID, "txn_month"]).size().rename("c").reset_index()
    mo["share"] = mo["c"] / mo.groupby(ID)["c"].transform("sum")
    mp = mo.pivot_table(index=ID, columns="txn_month", values="share", fill_value=0)
    keep = {11: "nov_share", 12: "dec_share", 1: "jan_share"}
    mp = mp[[c for c in keep if c in mp.columns]].rename(columns=keep)
    out = qp.join(mp, how="outer").reset_index()
    out["nov_dec_jan_share"] = out[[c for c in ["nov_share", "dec_share", "jan_share"]
                                    if c in out.columns]].fillna(0).sum(axis=1)
    return out


def _history(mf):
    h = mf.groupby(ID).agg(
        hist_total=("txn_count", "sum"), hist_mean=("txn_count", "mean"),
        hist_std=("txn_count", "std"), hist_max=("txn_count", "max"),
        hist_min=("txn_count", "min"), hist_median=("txn_count", "median"),
        hist_active_months=("txn_count", lambda x: (x > 0).sum()),
        hist_total_months=("txn_count", "count"), hist_amount_mean=("txn_sum", "mean"),
        hist_amount_cv=("amount_cv", "mean"), hist_debit_ratio=("debit_ratio", "mean"),
        hist_bal_mean=("bal_mean", "mean"), hist_weekend_mean=("weekend_ratio", "mean"),
        hist_month_end_mean=("month_end_ratio", "mean"), hist_turnover_mean=("turnover_ratio", "mean"),
        hist_amt_x_count=("amt_x_count", "mean"), hist_net_flow_mean=("net_flow", "mean"),
        hist_net_flow_std=("net_flow", "std")).reset_index()
    h["activity_rate"] = h["hist_active_months"] / (h["hist_total_months"] + 1)
    h["hist_txn_cv"] = h["hist_std"] / (h["hist_mean"] + 1e-9)
    h["annual_run_rate"] = h["hist_mean"] * 12
    h["hist_max_to_mean"] = h["hist_max"] / (h["hist_mean"] + 1e-9)
    last3 = mf[mf["months_ago"] <= 2].groupby(ID)["txn_count"].sum().rename("recent_3m_total")
    return h.merge(last3, on=ID, how="left")


def _ttype(T):
    recent = T[T["days_to_cutoff"] <= 180]
    ct = recent.groupby([ID, "TransactionTypeDescription"], observed=True).size().unstack(fill_value=0)
    ct = ct.div(ct.sum(axis=1) + 1e-9, axis=0)
    ct.columns = [f"ttype_{_san(c)}_share" for c in ct.columns]
    return ct.reset_index()


def _intervals(T):
    Ts = T.sort_values([ID, "TransactionDate"])
    Ts = Ts.assign(gap=Ts.groupby(ID)["TransactionDate"].diff().dt.days)
    g = Ts.groupby(ID)["gap"]
    out = pd.DataFrame({"gap_mean": g.mean(), "gap_median": g.median(),
                        "gap_std": g.std(), "gap_max": g.max()})
    out["gap_cv"] = out["gap_std"] / (out["gap_mean"] + 1e-9)
    out["expected_90d_count"] = 90.0 / (out["gap_mean"] + 1e-9)
    recent = Ts[Ts["days_to_cutoff"] <= 90]
    gr = recent.groupby(ID)["gap"]
    out["gap_mean_90d"] = gr.mean()
    out["gap_std_90d"] = gr.std()
    out["expected_next_from_90d"] = 90.0 / (gr.mean() + 1e-9)
    return out.reset_index()


def _accounts(T):
    acc = T.groupby(ID).agg(n_accounts=("AccountID", "nunique"),
                            n_txn_total=("TransactionDate", "count"))
    acc["txns_per_account"] = acc["n_txn_total"] / (acc["n_accounts"] + 1e-9)
    per_acc = T.groupby([ID, "AccountID"]).size()
    acc["top_account_share"] = per_acc.groupby(level=0).apply(lambda s: s.max() / s.sum())
    return acc.reset_index()


def _batch(T):
    recent = T[T["days_to_cutoff"] <= 180]
    ct = recent.groupby([ID, "TransactionBatchDescription"], observed=True).size().unstack(fill_value=0)
    ct = ct.div(ct.sum(axis=1) + 1e-9, axis=0)
    ct.columns = [f"batch_{_san(c)}_share" for c in ct.columns]
    return ct.reset_index()


def _extra(T, mf, cutoff_int):
    """Optional add-ons (rejected experiment, not wired into the final build): forward-looking
    closing-balance capacity, negative-balance / overdraft regime, batch-channel diversity,
    per-account dispersion."""
    m = mf.copy()
    rng = (m["bal_max"] - m["bal_min"]).replace(0, np.nan)
    m["bal_pos"] = ((m["bal_last"] - m["bal_min"]) / rng).clip(0, 1)
    m["bal_neg"] = (m["bal_last"] < 0).astype("int8")
    latest = m.sort_values("months_ago").groupby(ID).first()
    out = pd.DataFrame(index=pd.Index(m[ID].unique(), name=ID))
    out["bal_pos_latest"] = latest["bal_pos"]
    out["bal_pos_mean6"] = m[m["months_ago"] < 6].groupby(ID)["bal_pos"].mean()
    out["n_neg_bal_6m"] = m[m["months_ago"] < 6].groupby(ID)["bal_neg"].sum()
    out["n_neg_bal_12m"] = m[m["months_ago"] < 12].groupby(ID)["bal_neg"].sum()
    out["overdraft_recent"] = (m[m["months_ago"] < 3].groupby(ID)["bal_neg"].sum() > 0).astype("float")
    # balance level slope over 24 months (we already have 3/6/12 in _snap_mom)
    bl = m[m["months_ago"] < 24].sort_values([ID, "months_ago"])
    def _slope(g):
        x = -g["months_ago"].values.astype(float); x = x - x.mean()
        y = g["bal_last"].values.astype(float)
        return float((x * y).sum() / ((x ** 2).sum() + 1e-9)) if len(g) > 1 else 0.0
    out["bal_slope24"] = bl.groupby(ID).apply(_slope)
    # batch-channel diversity (recent 180d)
    rec = T[T["days_to_cutoff"] <= 180]
    bc = rec.groupby([ID, "TransactionBatchDescription"]).size()

    def _ent(s):
        p = s.values / s.values.sum()
        return float(-(p * np.log(p + 1e-12)).sum())
    out["batch_entropy"] = bc.groupby(level=0).apply(_ent)
    out["batch_distinct"] = bc.groupby(level=0).size()
    # per-account dispersion (how concentrated activity is across accounts)
    pa = T.groupby([ID, "AccountID"]).size()
    out["acc_cov"] = pa.groupby(level=0).agg(lambda s: s.std() / (s.mean() + 1e-9))
    return out.reset_index()


def _financials(cutoff):
    f = pd.read_parquet(DATA / "financials_features.parquet")
    f["RunDate"] = pd.to_datetime(f["RunDate"])
    f = f[f["RunDate"] <= cutoff].sort_values([ID, "RunDate"])
    agg = f.groupby(ID).agg(nii_mean=("NetInterestIncome", "mean"), nii_std=("NetInterestIncome", "std"),
                            nii_last=("NetInterestIncome", "last"), nir_mean=("NetInterestRevenue", "mean"),
                            nir_last=("NetInterestRevenue", "last"), n_fin=("NetInterestIncome", "count"),
                            n_products=("Product", "nunique")).reset_index()
    agg["nii_change"] = agg["nii_last"] - agg["nii_mean"]
    agg["nii_cv"] = agg["nii_std"].fillna(0) / (agg["nii_mean"].abs() + 1e-9)
    rec = f.groupby(ID).tail(3).groupby(ID).agg(nii_rec_mean=("NetInterestIncome", "mean")).reset_index()
    agg = agg.merge(rec, on=ID, how="left")
    agg["nii_momentum"] = agg["nii_rec_mean"] - agg["nii_mean"]
    return agg


def _demographics(cutoff):
    d = pd.read_parquet(DATA / "demographics_clean.parquet")
    d["BirthDate"] = pd.to_datetime(d["BirthDate"], errors="coerce")
    ref = cutoff.replace(day=1)
    d["age"] = (ref - d["BirthDate"]).dt.days // 365
    d.loc[(d["age"] < 16) | (d["age"] > 100), "age"] = np.nan
    d["age_sq"] = d["age"] ** 2
    d["age_bucket"] = pd.cut(d["age"], bins=[0, 25, 35, 50, 65, 200],
                             labels=[0, 1, 2, 3, 4]).astype("float")
    d["income_missing"] = d["AnnualGrossIncome"].isna().astype("int8")
    d["log_income"] = np.log1p(d["AnnualGrossIncome"].clip(lower=0))
    drop = ["BirthDate", "ResidentialCityName", "CountryCodeNationality"]
    return d.drop(columns=[c for c in drop if c in d.columns])


def _tenure(T, cutoff_int, ids):
    """Calendar months-on-book and true activity density as-of cutoff.

    The generic builder median-imputes absent customers, which makes a
    not-yet-onboarded / short-history customer indistinguishable from a dormant
    veteran - the defect that made earlier (sparse-history) anchors look
    off-distribution and capped the panel. months_on_book is CALENDAR months
    since the first ever transaction, NOT the
    active-month count that _history already has. Built over the full id list so
    absent customers get has_history=0 / months_on_book=0 (explicit zeros, never
    the median), letting the model gate on tenure instead of guessing.
    """
    g = T.groupby(ID)["ym_int"]
    info = pd.DataFrame({"first_ym": g.min(), "n_active_months": g.nunique()})
    out = pd.DataFrame({ID: pd.unique(np.asarray(ids))}).merge(
        info, left_on=ID, right_index=True, how="left")
    out["has_history"] = out["first_ym"].notna().astype("int8")
    out["months_on_book"] = (cutoff_int - out["first_ym"]).fillna(0).clip(lower=0).astype("int16")
    out["n_active_months_cal"] = out["n_active_months"].fillna(0).astype("int16")
    out["active_month_density"] = out["n_active_months_cal"] / (out["months_on_book"] + 1)
    return out.drop(columns=["first_ym", "n_active_months"])


def build_asof(cutoff_str: str, ids: pd.Series):
    """Return (X, y, feat_cols) for the given cutoff over the supplied customer ids.
    y = count of transactions in the 3 months after cutoff (raw count, >=1 kept)."""
    cutoff = pd.Timestamp(cutoff_str)
    cutoff_int = cutoff.to_period("M").ordinal
    T = _prep(cutoff)
    mf = _monthly(T, cutoff_int)   # compute ONCE (was rebuilt 4x per anchor - CPU + memory waste)
    parts = [_daily(T), _festive(mf),
             _recency(T, mf, cutoff), _snap_mom(mf),
             _seasonal(T), _history(mf), _ttype(T),
             _intervals(T), _accounts(T), _batch(T), _tenure(T, cutoff_int, ids)]
    # NOTE: _extra(...) (closing-balance position/neg-counts/slope24, batch entropy, account
    # CoV) was tested and REJECTED - L1 real-anchor CV 0.37667 vs base 0.37599 (worse).
    # Kept defined but NOT wired into the final build.
    del T, mf
    gc.collect()
    base = pd.DataFrame({ID: ids.values})
    for p in parts:
        base = base.merge(p, on=ID, how="left")
    base = base.merge(_financials(cutoff), on=ID, how="left")
    base = base.merge(_demographics(cutoff), on=ID, how="left")

    base = base.loc[:, ~base.columns.duplicated()]
    # encode object categoricals (label) consistently
    for c in [c for c in base.columns if base[c].dtype == "object" and c != ID]:
        cats = base[c].astype("string").fillna("NA")
        mp = {v: i for i, v in enumerate(sorted(cats.unique()))}
        base[c] = cats.map(mp).astype("int32")

    # target from post-cutoff window
    tgt = compute_target(cutoff)
    y = base[ID].map(tgt)

    feat = [c for c in base.columns if c != ID]
    # NOTE: NaN-native handling (passing missing straight to LightGBM) was tested and was
    # WORSE on the leaderboard: it regressed 0.37069 -> 0.377782 (+0.007). The per-build median
    # fill + 0-fill is empirically better here because 0 correctly encodes "no activity" for the
    # count/share features that dominate. Retained.
    for c in feat:
        if base[c].dtype.kind in "biufc":
            base[c] = base[c].fillna(base[c].median())
    base[feat] = base[feat].fillna(0)
    return base[feat], y, feat

## 4. The multi-anchor panel (the idea that worked)

The obvious setup is one row per customer, described as of 31 Oct 2015, predicting the next 3
months - 8,360 training rows. The problem (Section 1): with a single snapshot the model can quietly
**memorise those 8,360 specific people**, which does not transfer to disjoint strangers.

A **multi-anchor panel** fixes this. It is a standard rolling-origin technique for time-series: pick
many historical "anchor" dates, and at each one describe every customer using only what was known by
that date, with a target equal to their realized next-3-month count (which we can simply count from
the history, because that window is already in the past). One customer becomes many
(customer, anchor) rows across different seasons and activity levels, so the model is forced to learn
the general "given this recent history, expect this many next quarter" rule instead of customer
identity.

Our winning panel:
- **Real anchor 2015-10-31** (the month we are scored on): rows are the 8,360 **train** customers,
  labels from `Train.csv`. Test customers are excluded here (their future is the hidden target).
- **24 historical anchors** (month-ends 2013-10-31 to 2015-07-31): rows are **all 11,944 customers**
  (train and test), with targets counted directly from observed transactions. This is not leakage -
  the whole 3-month window at a historical anchor lies in the past relative to the Oct 2015 data
  cutoff - and it lets the model see the test customers' own historical behaviour, directly
  attacking the disjoint-customer problem.
- **Calendar features per anchor** (anchor month/year; whether the next 3 months include
  Nov/Dec/Jan; number of days and weekends) so the model knows which window it is predicting and
  seasonality is preserved across the stacked months.
- **Sample weights**: the real 2015-10 anchor is weighted 3.0, other Octobers 1.5, everything else
  1.0, because Oct->Jan is exactly the target shape.

The config below (the `CHAMP` LightGBM settings, the anchor list, the calendar features, and the
per-anchor frame builder) is inlined verbatim from the training code.

In [4]:
REAL = "2015-10-31"
HIST_ANCHORS = [d.strftime("%Y-%m-%d") for d in
                pd.date_range("2013-10-31", "2015-07-31", freq="ME")]
SEEDS = (42, 1, 7)
CHAMP = dict(objective="regression_l1", learning_rate=0.02, num_leaves=63,
             min_child_samples=40, subsample=0.7, subsample_freq=1, colsample_bytree=0.5,
             reg_alpha=2, reg_lambda=10, n_estimators=3000, n_jobs=-1, verbose=-1)


def _calendar(cutoff_str: str, n: int) -> pd.DataFrame:
    """Anchor + next-3m window calendar descriptors (constant within an anchor)."""
    c = pd.Timestamp(cutoff_str)
    win = pd.date_range(c + pd.offsets.MonthBegin(1), periods=3, freq="MS")
    months = set(win.month)
    end = c + pd.DateOffset(months=3)
    days = pd.date_range(c + pd.Timedelta(days=1), end, freq="D")
    row = {
        "anchor_month": c.month, "anchor_year": c.year,
        "has_nov_next3m": int(11 in months), "has_dec_next3m": int(12 in months),
        "has_jan_next3m": int(1 in months),
        "count_ndj_next3m": int(sum(m in (11, 12, 1) for m in win.month)),
        "n_days_next3m": len(days),
        "n_weekend_next3m": int((days.dayofweek >= 5).sum()),
    }
    return pd.DataFrame([row] * n).reset_index(drop=True)


def _anchor_frame(cutoff_str, ids, real_target=None):
    """Build one anchor's (X, y, weight, group) with calendar cols appended."""
    X, y, feat = build_asof(cutoff_str, ids)
    if real_target is not None:
        y = ids.map(real_target).reset_index(drop=True)
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y, float))
    cal = _calendar(cutoff_str, len(X))
    X = pd.concat([X.reset_index(drop=True), cal], axis=1)
    keep = y.notna() & (y >= 1)
    X, y = X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)
    grp = pd.Series(np.asarray(ids)[keep.values])
    is_oct = pd.Timestamp(cutoff_str).month == 10
    w = pd.Series(np.where(real_target is not None, 3.0, 1.5 if is_oct else 1.0), index=X.index)
    return X, y, w, grp

## 5. The model and the core finding

The learner is a single **LightGBM** gradient-boosted-tree regressor with the **L1 (absolute-error)
objective**, trained on the **log1p** of the target (the metric lives in log space), and heavily
regularised. Every hyperparameter (slow learning, humble trees, half the features/rows per tree,
strong L1/L2 penalties) is chosen to fight memorisation, because of the central empirical finding
from our own leaderboard experiments:

> **Adding same-distribution data helps the leaderboard; adding model capacity (extra features,
> heavier tuning, ensembles of correlated models) improves cross-validation but hurts the
> leaderboard.**

On a disjoint-customer problem, extra capacity just gives the model more room to memorise training
customers, which does not transfer. So after the panel was in place, the only remaining
leaderboard-safe lever was **variance reduction**: train the identical model over many random seeds
and average the predictions. That cannot overfit - it only cancels Monte-Carlo noise. We watched it
converge: 3 seeds -> public 0.37069; 24 seeds -> 0.369928; 36 seeds -> **0.369841** (the variance
floor). The winning file is the 36-seed average.

In [5]:
# The 36 seeds used for the winning submission (variance reduction only - identical model each time).
SEEDS_36 = (42, 1, 7, 13, 23, 99, 123, 777, 2024, 5, 17, 31,
            2, 3, 11, 19, 29, 37, 41, 53, 67, 101, 199, 256,
            61, 71, 73, 79, 83, 89, 97, 103, 107, 109, 113, 127)
assert len(set(SEEDS_36)) == 36


def build_allcust_panel():
    """All-customers multi-anchor panel: real anchor = train customers (labels from Train.csv);
    every historical anchor = ALL 11,944 customers, target counted from observed transactions."""
    train_ids = pd.read_csv(DATA / "Train.csv")[ID]
    test_ids = pd.read_csv(DATA / "Test.csv")[ID]
    all_ids = pd.concat([train_ids, test_ids], ignore_index=True)
    real_tgt = pd.read_csv(DATA / "Train.csv").set_index(ID)[TARGET]

    Xr, yr, wr, _ = _anchor_frame(REAL, train_ids, real_target=real_tgt)   # real anchor: train only
    ref = list(Xr.columns)
    blocks = [(Xr.astype("float32"), yr, wr)]
    for a in HIST_ANCHORS:                                                 # history: all customers
        Xa, ya, wa, _ = _anchor_frame(a, all_ids)
        blocks.append((Xa.reindex(columns=ref, fill_value=0.0).astype("float32"), ya, wa))
        print(f"  anchor {a}: {len(Xa)} rows", flush=True)
    X = pd.concat([b[0] for b in blocks], ignore_index=True).astype("float32")
    ylog = np.log1p(pd.concat([b[1] for b in blocks], ignore_index=True).values)
    w = pd.concat([b[2] for b in blocks], ignore_index=True).values
    del blocks; gc.collect()
    return X, ref, ylog, w

## 6. Reproduce the winning submission

Set `RUN_TRAINING = True` at the top and run all cells to rebuild the panel from raw data, train the
36 seeds, and write `submissions/panel_allcust_36seed_REPRO.csv`. The default (`False`) skips the
multi-hour rebuild and goes straight to verification against the committed file.

Memory note: the pipeline is tuned for a 16 GB machine - the transaction table is loaded
column-pruned and downcast, and models are freed with `gc` between fits.

In [6]:
if RUN_TRAINING:
    import time
    t0 = time.time()
    print("Building all-customers panel from raw data ...", flush=True)
    X, ref, ylog, w = build_allcust_panel()
    print(f"panel {X.shape} built in {(time.time()-t0)/60:.1f} min", flush=True)

    test_ids = pd.read_csv(DATA / "Test.csv")[ID]
    Xte, _, _ = build_asof(REAL, test_ids)
    Xte = pd.concat([Xte.reset_index(drop=True), _calendar(REAL, len(Xte))], axis=1)
    Xte = Xte.reindex(columns=ref, fill_value=0.0).astype("float32")

    tp = np.zeros(len(Xte))
    for i, s in enumerate(SEEDS_36, 1):
        m = lgb.LGBMRegressor(**{**CHAMP, "random_state": s})
        m.fit(X, ylog, sample_weight=w)
        tp += m.predict(Xte) / len(SEEDS_36)         # average in log space
        del m; gc.collect()
        print(f"  seed {i}/36 done", flush=True)
    out = write_submission(test_ids, np.clip(np.expm1(tp), 0, None), "panel_allcust_36seed_REPRO.csv")
    print(f"wrote {out} in {(time.time()-t0)/60:.1f} min total", flush=True)
else:
    print("RUN_TRAINING is False - skipping the rebuild. Verification is in the next cell.")

RUN_TRAINING is False - skipping the rebuild. Verification is in the next cell.


## 7. Verification

Confirms the committed winning file is well-formed, and - if you ran the reproduction above -
compares the freshly-built file to it.

We ran that full end-to-end reproduction on a clean environment. The freshly-built file matched the
committed submission with **correlation 0.99991**, median per-prediction difference **0.004** and
RMS difference **0.016** (log space); a handful of customers differ more (max 0.46) because LightGBM
CPU training with multiple threads is not bit-identical across runs, and small per-seed differences
compound for a few customers over a deep 3,000-tree ensemble. The aggregate is far below the
leaderboard's resolution, so a fresh run **reproduces the leaderboard score** even though it is not
byte-for-byte identical. The committed `submissions/panel_allcust_36seed.csv` is the exact file that
scored on the leaderboard. (For strict bit-exactness, set `num_threads=1` and `deterministic=True` -
at a large cost in runtime, and it no longer matches the original multi-threaded file.)

In [7]:
committed = pd.read_csv("submissions/panel_allcust_36seed.csv")
print("committed winning submission: submissions/panel_allcust_36seed.csv (public LB 0.369841)")
print(f"  rows: {len(committed)}  columns: {list(committed.columns)}")
print(f"  stored as log1p(count): min={committed[TARGET].min():.3f} max={committed[TARGET].max():.3f}"
      f"  -> {np.expm1(committed[TARGET].min()):.1f} .. {np.expm1(committed[TARGET].max()):.0f} txns")
print(f"  all finite & non-negative: {np.isfinite(committed[TARGET]).all() and (committed[TARGET]>=0).all()}")

repro = Path("submissions/panel_allcust_36seed_REPRO.csv")
if repro.exists():
    a = committed.set_index(ID)[TARGET]
    b = pd.read_csv(repro).set_index(ID)[TARGET].reindex(a.index)
    d = np.abs(a.values - b.values)
    print(f"\nreproduced vs committed: corr={np.corrcoef(a.values, b.values)[0,1]:.8f}  "
          f"max|delta|={d.max():.3e}  mean|delta|={d.mean():.3e}")
committed.head()

committed winning submission: submissions/panel_allcust_36seed.csv (public LB 0.369841)
  rows: 3584  columns: ['UniqueID', 'next_3m_txn_count']
  stored as log1p(count): min=1.256 max=7.953  -> 2.5 .. 2843 txns
  all finite & non-negative: True


,UniqueID,next_3m_txn_count
0,6b62ce75-9823-4de6-ba7b-8b2b199df239,2.980102
1,e193e600-a706-4bc6-8597-d5d6fb171ab5,1.302093
2,8fd44803-12ed-46ab-a146-8496d95d1b13,3.812413
3,12606376-113f-4c90-94b9-65f64f9fa8c7,1.712626
4,da070817-27ed-44b2-bc05-e817ea311519,5.599711


## 8. What we tried - what worked and what did not

We treated the leaderboard like a lab: each probe changed one thing against a clean baseline. The
most useful result is negative - once the panel was in place, almost nothing else helped, which is
itself the finding.

**Worked (and is in the final solution):**
- Multi-anchor panel vs single cross-section - the core lift.
- Including all customers (not just train) at historical anchors - best single model.
- Seed averaging 3 -> 36 - pure denoise, 0.37069 -> 0.369841.
- log1p submission, L1 objective, heavy regularisation - foundational.

**Did not work (cross-validation said yes, leaderboard said no):**
- EWMA-decay recency features on the clean base: better CV, leaderboard 0.373984 (worse).
- Heavier tuning / a regularisation sweep: looked like a win at 3 seeds (0.369992), but at 24 seeds
  it was 0.370582 - the "win" was seed luck.
- Passing missing values as NaN instead of the median/0 fill: 0.377782 (+0.007 worse).
- Closing-balance "capacity to spend" features: 0.374-0.378 (worse).
- Deeper tenure + raw-space averaging: 0.375698 (worse).

**Deliberately not pursued:** blending several correlated boosted-tree models (once each model was
0.9+ correlated, blending stopped helping); and a fitted saturation-growth-curve baseline plus
residual (a different modelling approach), which we judged a different-method gap rather than a
missing feature, and out of scope for the final days.

**The transferable lesson:** on a disjoint-group problem, trust the direction of the leaderboard
over local cross-validation, expand *data* rather than *capacity*, and denoise with seeds before
believing a small gain.

## 9. How to run, and notes

1. Put the competition data in `data/` (not bundled - see `data/README.md`): `Train.csv`,
   `Test.csv`, `SampleSubmission.csv`, `transactions_features.parquet`,
   `financials_features.parquet`, `demographics_clean.parquet`.
2. `pip install -r requirements.txt` (Python 3.12; lightgbm, pandas, numpy, scikit-learn, pyarrow).
3. Run all cells. Default (`RUN_TRAINING = False`) verifies the committed submission in seconds; set
   `RUN_TRAINING = True` to rebuild from scratch.

**Environment used:** Apple M5, 10-core CPU, 16 GB RAM, macOS, Python 3.12. CPU only, no GPU.
**Runtime (full reproduction):** ~107 minutes end-to-end on that machine - 10.7 min to build the
panel/features + ~96 min for the 36 LightGBM seed fits + inference. CPU only.
**Data:** only the competition-provided data was used - no external datasets.
**Method note:** the multi-anchor panel is a standard rolling-origin technique for time-series
forecasting; all findings above are from our own leaderboard experiments on this competition.
**Licence:** MIT (see `LICENSE`).